# 04 — Khoảng dự đoán cho mô hình đã khóa (calibration)

Mô hình cuối (notebook 03, C64–C68) chỉ cho **một con số**. Notebook này thêm **khoảng dao động 80%**: khoảng mà giá thật nằm trong đó ở khoảng 8/10 căn mô hình chưa từng thấy.

**Phương pháp:** split conformal prediction. Phương pháp này không cần giả định phân phối sai số. Nó chỉ dùng sai số của mô hình trên một tập **chưa tham gia huấn luyện**, ở đây là tập calibration (3.023 dòng, chia sẵn từ notebook 01). Nếu dữ liệu mới giống dữ liệu calibration (cùng phân phối), khoảng tạo ra phủ giá thật với xác suất tối thiểu 80%.

**Quy trình chống rò rỉ:**
1. Mô hình giữ nguyên, không refit. File `.pkl` được nạp qua `app/core/model_interface.py`, tức là có kiểm tra SHA-256, Python và thư viện như ứng dụng.
2. Khoảng chỉ được **học trên calibration** (C03).
3. Phương pháp được **chọn trên validation** theo quy tắc viết trước khi xem kết quả (C04).
4. Phương pháp được **khóa và lưu** (C05), sau đó mới đo **một lần trên test** (C06). Tập test đã được dùng ở notebook 03 (C67) để đo sai số điểm; độ phủ của khoảng chỉ được đo ở đây, sau khi đã khóa.

Kết quả lưu ở `experiments/calibration/prediction_interval.json`. Ứng dụng Streamlit và bản web đọc file này.

In [1]:
# C01 — Mô hình đã khóa, các tập dữ liệu và dự đoán (chưa đọc tập test)
import json
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "app"))
from core import model_interface  # noqa: E402  nạp mô hình đúng như ứng dụng

loaded = model_interface.load_project_model()
assert loaded.model is not None, loaded.problem
model, manifest = loaded.model, loaded.manifest

SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "splits_v2"
OUT_PATH = PROJECT_ROOT / "experiments" / "calibration" / "prediction_interval.json"
raw_address = pd.read_csv(PROJECT_ROOT / "data" / "vietnam_housing_dataset.csv", usecols=["Address"])["Address"]


def load_split(name):
    split = pd.read_csv(SPLIT_DIR / f"{name}.csv")
    frame = split[model.features].copy()
    frame["Address"] = raw_address.iloc[split["row_id"].to_numpy()].to_numpy()
    return split, frame


train, train_frame = load_split("train")
# Tin trùng: cùng địa chỉ, diện tích và giá với một tin trong train (cùng định nghĩa với notebook 03, C67).
train_keys = set(zip(train_frame["Address"], train["Area"], train["Price"]))
PRICE_RANGE = (float(train["Price"].min()), float(train["Price"].max()))


def predictions(name):
    split, frame = load_split(name)
    return pd.DataFrame({
        "y": split["Price"].to_numpy(),
        "pred": model.predict(frame),
        "twin": [key in train_keys for key in zip(frame["Address"], split["Area"], split["Price"])],
    })


data = {name: predictions(name) for name in ["calibration", "validation"]}
for name, d in data.items():
    print(f"{name}: {len(d)} dòng, MAE {np.abs(d.y - d.pred).mean():.3f} tỷ, "
          f"tin trùng với train {d.twin.mean():.1%}")
print("Khoảng giá trong train:", PRICE_RANGE, "| mô hình:", manifest["sha256"][:12])

2026-10-20 22:46:50.721 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-20 22:46:50.738 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


calibration: 3023 dòng, MAE 0.871 tỷ, tin trùng với train 10.7%
validation: 4535 dòng, MAE 0.880 tỷ, tin trùng với train 10.5%
Khoảng giá trong train: (1.0, 11.5) | mô hình: abcde93a531f


## Sai số trên calibration theo mức giá dự đoán

Nếu sai số đều nhau ở mọi mức giá thì chỉ cần một độ rộng chung. Bảng dưới chia calibration thành 5 nhóm bằng nhau theo giá dự đoán.

In [2]:
# C02 — Sai số (giá thật − giá dự đoán) trên calibration, theo 5 nhóm giá dự đoán
cal = data["calibration"].assign(residual=lambda d: d.y - d.pred)
cal["group"] = pd.qcut(cal["pred"], 5)
residual_summary = cal.groupby("group", observed=True)["residual"].agg(
    n="size",
    mean="mean",
    MAE=lambda r: r.abs().mean(),
    q10=lambda r: r.quantile(0.10),
    q90=lambda r: r.quantile(0.90),
)
display(residual_summary.round(2))

,n,mean,MAE,q10,q90
group,,,,,
"(0.817, 4.15]",605,-0.13,0.62,-1.14,0.70
"(4.15, 5.473]",604,-0.02,0.85,-1.37,1.37
"(5.473, 6.573]",605,-0.03,0.96,-1.47,1.53
"(6.573, 7.532]",604,0.10,1.04,-1.48,1.98
"(7.532, 10.492]",605,-0.00,0.88,-1.49,1.39


## Hai cách tạo khoảng, chỉ học trên calibration

Ký hiệu $\hat y$ là giá dự đoán, $r = y - \hat y$ là sai số, $\alpha = 0{,}2$ (khoảng 80%), $n$ là số dòng dùng để học.

* **Khoảng đều** (split conformal đối xứng): $[\hat y - q,\ \hat y + q]$. Ở đây $q$ là giá trị thứ $\lceil (n+1)(1-\alpha) \rceil$ của $|r|$ đã sắp xếp. Mọi căn có cùng độ rộng.
* **Khoảng theo nhóm giá** (Mondrian split conformal, hai đầu riêng): chia calibration thành 5 nhóm theo $\hat y$ (ranh giới là các phân vị 20/40/60/80% của $\hat y$). Trong mỗi nhóm, khoảng là $[\hat y + q_{lo},\ \hat y + q_{hi}]$, với $q_{lo}$ là giá trị thứ $\lfloor (n+1)\alpha/2 \rfloor$ và $q_{hi}$ là giá trị thứ $\lceil (n+1)(1-\alpha/2) \rceil$ của $r$ đã sắp xếp. Mỗi đầu bỏ ra tối đa 10%, nên độ phủ trong từng nhóm tối thiểu 80%. Hai đầu riêng cho phép khoảng lệch về phía mô hình hay đoán sai.

Cả hai đều được **cắt về khoảng giá có trong train** (1–11,5 tỷ), vì giá thật trong dữ liệu không nằm ngoài khoảng này. Khoảng luôn chứa giá ước tính.

In [3]:
# C03 — Học hai loại khoảng trên calibration
ALPHA = 0.2
N_GROUPS = 5


def order_stat(sorted_values, rank):
    """Giá trị thứ `rank` (đếm từ 1) của mảng đã sắp xếp."""
    assert 1 <= rank <= len(sorted_values), "quá ít dòng cho mức alpha này"
    return float(sorted_values[rank - 1])


def fit_constant(pred, y):
    scores = np.sort(np.abs(y - pred))
    return {"method": "constant", "q": order_stat(scores, int(np.ceil((len(scores) + 1) * (1 - ALPHA))))}


def fit_grouped(pred, y):
    edges = np.quantile(pred, np.arange(1, N_GROUPS) / N_GROUPS)
    group = np.searchsorted(edges, pred, side="right")
    groups = []
    for g in range(N_GROUPS):
        r = np.sort((y - pred)[group == g])
        n = len(r)
        groups.append({
            "n": int(n),
            "q_lo": order_stat(r, int(np.floor((n + 1) * ALPHA / 2))),
            "q_hi": order_stat(r, int(np.ceil((n + 1) * (1 - ALPHA / 2)))),
        })
    return {"method": "grouped", "edges": [float(e) for e in edges], "groups": groups}


def interval(params, pred):
    """(low, high) cho từng giá dự đoán: cắt về khoảng giá của train, luôn chứa giá dự đoán."""
    pred = np.asarray(pred, dtype=float)
    if params["method"] == "constant":
        low, high = pred - params["q"], pred + params["q"]
    else:
        group = np.searchsorted(params["edges"], pred, side="right")
        low = pred + np.array([params["groups"][g]["q_lo"] for g in group])
        high = pred + np.array([params["groups"][g]["q_hi"] for g in group])
    low = np.minimum(np.clip(low, *PRICE_RANGE), pred)
    high = np.maximum(np.clip(high, *PRICE_RANGE), pred)
    return low, high


cal = data["calibration"]
candidates = {
    "constant": fit_constant(cal["pred"].to_numpy(), cal["y"].to_numpy()),
    "grouped": fit_grouped(cal["pred"].to_numpy(), cal["y"].to_numpy()),
}
print(f"Khoảng đều: ±{candidates['constant']['q']:.3f} tỷ")
display(pd.DataFrame(candidates["grouped"]["groups"]).assign(
    from_pred=[-np.inf, *candidates["grouped"]["edges"]], to_pred=[*candidates["grouped"]["edges"], np.inf]
).round(3))

Khoảng đều: ±1.419 tỷ


,n,q_lo,q_hi,from_pred,to_pred
0,605,-1.145,0.698,-inf,4.150
1,604,-1.376,1.387,4.150,5.473
2,605,-1.506,1.540,5.473,6.573
3,604,-1.504,1.987,6.573,7.532
4,605,-1.526,1.405,7.532,inf


## Chọn phương pháp trên validation

**Quy tắc, viết trước khi chạy C04:** chọn phương pháp có độ rộng trung bình nhỏ nhất trong các phương pháp thoả hai điều kiện trên validation:
* độ phủ tổng thể từ 78% trở lên;
* không nhóm giá nào có độ phủ dưới 70%.

Nhóm giá dùng chung ranh giới của phương pháp theo nhóm, để hai phương pháp được so trên cùng các nhóm. Độ phủ trên **tin không trùng với train** được báo riêng, vì khoảng 10% dòng validation có bản trùng trong train, mà các dòng này dễ đoán hơn.

In [4]:
# C04 — Đánh giá trên validation và chọn theo quy tắc
GROUP_EDGES = candidates["grouped"]["edges"]


def evaluate(params, d):
    low, high = interval(params, d["pred"])
    inside = (d["y"] >= low) & (d["y"] <= high)
    group = np.searchsorted(GROUP_EDGES, d["pred"], side="right")
    return {
        "coverage": float(inside.mean()),
        "coverage_new_listings": float(inside[~d["twin"]].mean()),
        "group_coverage": [float(inside[group == g].mean()) for g in range(N_GROUPS)],
        "mean_width": float((high - low).mean()),
        "median_width": float(np.median(high - low)),
    }


val = data["validation"]
validation_results = {name: evaluate(params, val) for name, params in candidates.items()}
table = pd.DataFrame(validation_results).T
table["min_group_coverage"] = table["group_coverage"].map(min)
display(table.drop(columns="group_coverage").astype(float).round(3))
display(pd.DataFrame({name: r["group_coverage"] for name, r in validation_results.items()},
                     index=[f"nhóm {g + 1}" for g in range(N_GROUPS)]).round(3))

eligible = table[(table["coverage"] >= 0.78) & (table["min_group_coverage"] >= 0.70)]
assert len(eligible), "không phương pháp nào đạt quy tắc"
CHOICE = eligible["mean_width"].astype(float).idxmin()
print("Chọn:", CHOICE)

,coverage,coverage_new_listings,mean_width,median_width,min_group_coverage
constant,0.807,0.792,2.824,2.837,0.734
grouped,0.808,0.793,2.811,2.931,0.804


,constant,grouped
nhóm 1,0.907,0.804
nhóm 2,0.814,0.806
nhóm 3,0.779,0.806
nhóm 4,0.734,0.813
nhóm 5,0.802,0.811


Chọn: grouped


In [5]:
# C05 — Khóa phương pháp đã chọn (trước khi đọc tập test)
record = {
    "created": str(date.today()),
    "notebook": "notebooks/04_calibration.ipynb",
    "model": {"path": model_interface.MODEL_LABEL + "/house_price_ensemble.pkl", "sha256": manifest["sha256"]},
    "target_coverage": 1 - ALPHA,
    "price_range": list(PRICE_RANGE),
    "fit_on": {"split": "data/processed/splits_v2/calibration.csv", "rows": int(len(cal))},
    "method": CHOICE,
    "params": candidates[CHOICE],
    "rules": {
        "interval": "[pred + q_lo, pred + q_hi] theo nhóm giá dự đoán (grouped) hoặc pred ± q (constant); "
                    "cắt về price_range; luôn chứa pred",
        "selection": "độ rộng trung bình nhỏ nhất trong các phương pháp có độ phủ validation >= 78% và mọi nhóm >= 70%",
    },
    "validation": {"rows": int(len(val)), "twin_share": float(val["twin"].mean()), "results": validation_results},
    "test": None,
}
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(json.dumps(record, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
print("Đã lưu", OUT_PATH.relative_to(PROJECT_ROOT))

Đã lưu experiments\calibration\prediction_interval.json


In [6]:
# C06 — Đo một lần trên tập test, sau khi đã khóa ở C05
assert json.loads(OUT_PATH.read_text(encoding="utf-8"))["test"] is None, "test đã được đo; không chạy lại"
test = predictions("test")
test_result = evaluate(record["params"], test)
record["test"] = {
    "rows": int(len(test)),
    "twin_share": float(test["twin"].mean()),
    **test_result,
    "note": "Tập test đã được dùng một lần ở notebook 03 (C67) để đo sai số điểm; "
            "độ phủ của khoảng được đo một lần ở đây, sau khi phương pháp đã khóa ở C05.",
}
OUT_PATH.write_text(json.dumps(record, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
display(pd.Series({k: v for k, v in test_result.items() if k != "group_coverage"}).round(3))
print("Độ phủ theo nhóm giá:", [round(c, 3) for c in test_result["group_coverage"]])

coverage                 0.800
coverage_new_listings    0.783
mean_width               2.813
median_width             2.931
dtype: float64

Độ phủ theo nhóm giá: [0.816, 0.801, 0.757, 0.823, 0.807]


In [7]:
# C07 — Ví dụ: khoảng cho các căn dùng để kiểm tra ứng dụng
examples = [
    {"City": "Hà Nội", "District": "Cầu Giấy", "Ward": "Phường Dịch Vọng", "Area": 60.0, "Floors": 4},
    {"City": "Hồ Chí Minh", "District": "Gò Vấp", "Ward": "Phường 12", "Street": "Đường Quang Trung", "Area": 54.0,
     "Frontage": 4.1, "Access Road": 3.5, "Floors": 2, "Bedrooms": 2, "Bathrooms": 3,
     "House direction": "Tây - Nam", "Legal status": "Have certificate", "Furniture state": "Full"},
]
rows = []
for values in examples:
    pred = float(model.predict(model_interface.model_frame(values))[0])
    low, high = interval(record["params"], [pred])
    rows.append({"căn": f"{values['Ward']}, {values['District']}, {values['Area']:g} m²", "giá ước tính": round(pred, 2),
                 "khoảng 80%": f"{low[0]:.2f} – {high[0]:.2f}"})
display(pd.DataFrame(rows))

,căn,giá ước tính,khoảng 80%
0,"Phường Dịch Vọng, Cầu Giấy, 60 m²",8.93,7.40 – 10.33
1,"Phường 12, Gò Vấp, 54 m²",5.02,3.64 – 6.41


## Kết luận

* **Phương pháp chọn: khoảng theo nhóm giá** (Mondrian split conformal, hai đầu riêng, 5 nhóm theo giá dự đoán), học trên 3.023 dòng calibration.
* **Validation (4.535 dòng):** độ phủ 80,8%; trên tin không trùng với train 79,3%; mọi nhóm giá đều ở mức 80,4–81,3%. Độ rộng trung bình 2,81 tỷ (trung vị 2,93 tỷ).
* **Khoảng đều** ($\pm 1{,}42$ tỷ) có độ phủ tổng thể tương đương (80,7%) và độ rộng gần như bằng (2,82 tỷ). Nhưng nó chỉ phủ 73% ở nhóm 6,6–7,5 tỷ và tới 91% ở nhóm dưới 4,15 tỷ: hẹp quá ở chỗ mô hình hay sai, rộng quá ở chỗ mô hình đoán tốt. Khoảng theo nhóm sửa được điều này, vì nhà rẻ có khoảng hẹp và lệch xuống, còn nhóm 6,6–7,5 tỷ có đầu trên rộng hơn.
* **Test (đo một lần, sau khi khóa):** độ phủ 80,0%; trên tin không trùng với train 78,3%; theo nhóm giá 75,7–82,3%; độ rộng trung bình 2,81 tỷ.
* **Ý nghĩa:** khoảng 80% rộng khoảng 2,8 tỷ, tức gần một nửa giá trung vị (5,9 tỷ). Con số này phản ánh đúng mức bất định của mô hình (MAE 0,89 tỷ, $R^2$ 0,70); ứng dụng nên hiển thị khoảng cùng với giá.
* **Giới hạn:**
  * Độ phủ 80% là trung bình trên nhiều căn, kể cả trong từng nhóm giá; nó không đảm bảo cho từng căn cụ thể.
  * Bảo đảm lý thuyết cần dữ liệu mới giống dữ liệu năm 2024.
  * Tin đăng trùng làm độ phủ hơi lạc quan: với tin mới, độ phủ còn khoảng 78–79%.
  * Khoảng được cắt về 1–11,5 tỷ, vì mô hình không có cơ sở ngoài khoảng này.

Ứng dụng Streamlit (`app/core/model_interface.py`) và bản web (`web/public/model.js`) đọc `experiments/calibration/prediction_interval.json`. Cả hai chỉ dùng file này khi mã SHA-256 ghi trong đó khớp với mô hình đang chạy.